# Nifty Options Reinforcement Learning PPO Trainer

This Jupyter Notebook is designed to run on Google Colab using the **VS Code Colab Extension**.

### How to run via VS Code Colab Extension:
1. Open this `train_rl_colab.ipynb` file in VS Code.
2. Click on the **Select Kernel** button in the top-right corner of the notebook editor.
3. Select **Google Colab** from the kernel list. If prompted, sign in with your Google account.
4. Follow the cells below to install dependencies, upload the dataset, and train the agent.

### Step 1: Install Dependencies
This cell will install Stable Baselines3, Gymnasium, and other required libraries in the Google Colab environment.

In [ ]:
!pip install -q gymnasium stable-baselines3 xgboost scikit-learn pandas numpy

### Step 2: Upload Datasets
Run this cell to upload `nifty_rl_data.zip` from your local machine (located in your local `E:\\nse\\` folder) to the Google Colab workspace.

In [ ]:
from google.colab import files
import os
import zipfile

if not os.path.exists('nifty_rl_data.zip'):
    print("Select 'nifty_rl_data.zip' from your local E:\\nse\\ folder to upload:")
    uploaded = files.upload()
    
if os.path.exists('nifty_rl_data.zip'):
    with zipfile.ZipFile('nifty_rl_data.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Data extracted successfully!")
else:
    print("Error: nifty_rl_data.zip was not uploaded.")

### Step 3: Load Data and Feature Engineering
This cell loads options and spot data and engineers the exact 7 mathematical features.

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PREM_LOW = 100.0
PREM_HIGH = 600.0
TARGET_PTS = 10.0
STOP_PTS = 5.0
QTY = 25
MAX_CANDLES = 12

print("Loading datasets...")
opts = pd.read_csv('NIFTY_options_60d.csv')
spot = pd.read_csv('NIFTY_spot_60d.csv')
try:
    vix = pd.read_csv('nifty_vix_1y_5min.csv')
except:
    vix = pd.DataFrame({'timestamp': spot['timestamp'], 'close': 15.0})

opts['timestamp'] = pd.to_datetime(opts['timestamp']).dt.tz_localize(None)
spot['timestamp'] = pd.to_datetime(spot['timestamp']).dt.tz_localize(None)
vix['timestamp'] = pd.to_datetime(vix['timestamp']).dt.tz_localize(None)

print("Engineering features...")
s = spot.copy()
s['H'] = s['high']; s['L'] = s['low']; s['O'] = s['open']; s['C'] = s['close']
s['range'] = s['H'] - s['L']
s['atr_5'] = s['range'].rolling(5).mean()
s['atr_20'] = s['range'].rolling(20).mean()
s['VCS'] = np.where(s['atr_20'] > 0, s['atr_5'] / s['atr_20'], 1.0)
s['TFS_bull'] = (s['C'] - s['L']) / s['range'].replace(0, 1)

upper = s['H'] - s[['O','C']].max(axis=1)
lower = s[['O','C']].min(axis=1) - s['L']
s['WAC'] = np.where(s['range'] > 0, (upper - lower) / s['range'], 0)

s['PDV_3'] = (s['C'] - s['C'].shift(3)) / s['atr_5'].replace(0, 1)
s['PDV_5'] = (s['C'] - s['C'].shift(5)) / s['atr_5'].replace(0, 1)

vix_s = vix.rename(columns={'close':'vix_val'}).sort_values('timestamp')
s = pd.merge_asof(s.sort_values('timestamp'), vix_s[['timestamp', 'vix_val']], on='timestamp', direction='backward')
s['vix_norm'] = s['vix_val'] / 20.0
s = s.dropna().reset_index(drop=True)

opts = opts.rename(columns={'close': 'close_opt'})
opts = opts.sort_values('expiry')

opt_lookup = {}
for _, row in opts.iterrows():
    key = (row['timestamp'], row['strike'], row['opt_type'])
    if key not in opt_lookup:
        opt_lookup[key] = (row['close_opt'], row['expiry'])

atm_ce_close, atm_pe_close = [], []
atm_ce_strike, atm_pe_strike = [], []
atm_ce_expiry, atm_pe_expiry = [], []

for idx, row in s.iterrows():
    S = row['C']
    K = round(S / 50.0) * 50.0
    t = row['timestamp']
    ce_key = (t, K, 'CE')
    pe_key = (t, K, 'PE')
    
    if ce_key in opt_lookup:
        atm_ce_close.append(opt_lookup[ce_key][0])
        atm_ce_expiry.append(opt_lookup[ce_key][1])
    else:
        atm_ce_close.append(np.nan)
        atm_ce_expiry.append(None)
        
    if pe_key in opt_lookup:
        atm_pe_close.append(opt_lookup[pe_key][0])
        atm_pe_expiry.append(opt_lookup[pe_key][1])
    else:
        atm_pe_close.append(np.nan)
        atm_pe_expiry.append(None)
        
    atm_ce_strike.append(K)
    atm_pe_strike.append(K)
    
s['atm_ce_close'] = atm_ce_close
s['atm_pe_close'] = atm_pe_close
s['atm_ce_strike'] = atm_ce_strike
s['atm_pe_strike'] = atm_pe_strike
s['atm_ce_expiry'] = atm_ce_expiry
s['atm_pe_expiry'] = atm_pe_expiry

print(f"Data preprocessing complete. Total spot candles: {len(s)}")

### Step 4: Define the Gymnasium Environment
We create a custom environment that models the Nifty F&O trading logic.

In [ ]:
import gymnasium as gym
from gymnasium import spaces

class NiftyOptionsTradingEnv(gym.Env):
    def __init__(self, df, features_cols, transaction_cost=0.5):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.features_cols = features_cols
        self.transaction_cost = transaction_cost
        
        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(len(self.features_cols) + 4,),
            dtype=np.float32
        )
        self.reset()
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 20
        self.position = 0
        self.entry_premium = 0.0
        self.held_strike = 0.0
        self.held_type = None
        self.held_expiry = None
        self.entry_step = 0
        
        obs = self._get_obs()
        return obs, {}
        
    def _get_obs(self):
        row = self.df.iloc[self.current_step]
        feat_vals = row[self.features_cols].values.astype(np.float32)
        
        current_pnl = 0.0
        if self.position != 0:
            current_pnl = self._get_held_option_premium() - self.entry_premium
            
        atm_ce = row['atm_ce_close'] if not pd.isna(row['atm_ce_close']) else 50.0
        atm_pe = row['atm_pe_close'] if not pd.isna(row['atm_pe_close']) else 50.0
        
        obs = np.concatenate([
            feat_vals,
            np.array([float(self.position), current_pnl, atm_ce, atm_pe], dtype=np.float32)
        ])
        return obs
        
    def _get_held_option_premium(self):
        row = self.df.iloc[self.current_step]
        key = (row['timestamp'], self.held_strike, self.held_type)
        return opt_lookup.get(key, (self.entry_premium, None))[0]
        
    def step(self, action):
        reward = 0.0
        terminated = False
        truncated = False
        
        row = self.df.iloc[self.current_step]
        
        trade_info = None
        
        if self.position == 0:
            if action == 1 and not pd.isna(row['atm_ce_close']) and PREM_LOW <= row['atm_ce_close'] <= PREM_HIGH:
                self.position = 1
                self.entry_premium = row['atm_ce_close']
                self.held_strike = row['atm_ce_strike']
                self.held_type = 'CE'
                self.held_expiry = row['atm_ce_expiry']
                self.entry_step = self.current_step
                reward -= self.transaction_cost
            elif action == 2 and not pd.isna(row['atm_pe_close']) and PREM_LOW <= row['atm_pe_close'] <= PREM_HIGH:
                self.position = 2
                self.entry_premium = row['atm_pe_close']
                self.held_strike = row['atm_pe_strike']
                self.held_type = 'PE'
                self.held_expiry = row['atm_pe_expiry']
                self.entry_step = self.current_step
                reward -= self.transaction_cost
        else: 
            current_premium = self._get_held_option_premium()
            pnl = current_premium - self.entry_premium
            
            if pnl >= TARGET_PTS:
                reward += TARGET_PTS - self.transaction_cost
                self.position = 0
                trade_info = {
                    'pnl': TARGET_PTS - 2 * self.transaction_cost,
                    'type': self.held_type,
                    'entry_premium': self.entry_premium,
                    'exit_premium': current_premium,
                    'exit_reason': 'target_hit'
                }
            elif pnl <= -STOP_PTS:
                reward += -STOP_PTS - self.transaction_cost
                self.position = 0
                trade_info = {
                    'pnl': -STOP_PTS - 2 * self.transaction_cost,
                    'type': self.held_type,
                    'entry_premium': self.entry_premium,
                    'exit_premium': current_premium,
                    'exit_reason': 'stop_loss'
                }
            elif self.current_step - self.entry_step >= MAX_CANDLES:
                reward += pnl - self.transaction_cost
                self.position = 0
                trade_info = {
                    'pnl': pnl - 2 * self.transaction_cost,
                    'type': self.held_type,
                    'entry_premium': self.entry_premium,
                    'exit_premium': current_premium,
                    'exit_reason': 'max_candles'
                }
                
        self.current_step += 1
        if self.current_step >= len(self.df) - 1:
            terminated = True
            if self.position != 0:
                current_premium = self._get_held_option_premium()
                pnl = current_premium - self.entry_premium
                reward += pnl - self.transaction_cost
                self.position = 0
                trade_info = {
                    'pnl': pnl - 2 * self.transaction_cost,
                    'type': self.held_type,
                    'entry_premium': self.entry_premium,
                    'exit_premium': current_premium,
                    'exit_reason': 'termination'
                }
                
        obs = self._get_obs()
        info = {}
        if trade_info is not None:
            info['trade'] = trade_info
            
        return obs, reward, terminated, truncated, info

### Step 5: Train the PPO Reinforcement Learning Agent
We use Stable Baselines3 to train our agent on 70% chronological training data with a 15% validation split monitoring callback.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
import os
import shutil

train_split = int(len(s) * 0.70)
val_split = int(len(s) * 0.85)

train_df = s.iloc[:train_split]
val_df = s.iloc[train_split:val_split]
test_df = s.iloc[val_split:]

features_cols = ['VCS', 'TFS_bull', 'WAC', 'PDV_3', 'PDV_5', 'vix_norm', 'atr_5']

train_env = NiftyOptionsTradingEnv(train_df, features_cols)
val_env = NiftyOptionsTradingEnv(val_df, features_cols)
test_env = NiftyOptionsTradingEnv(test_df, features_cols)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training PPO Agent on hardware: {device.upper()}")

# Simplify neural network structure to prevent overfitting (regularization)
policy_kwargs = dict(
    net_arch=dict(pi=[32, 32], vf=[32, 32])
)

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=0.00015, # Lower learning rate for smoother convergence
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.02,        # Increase entropy coefficient to encourage exploration
    policy_kwargs=policy_kwargs,
    verbose=1,
    device=device
)

# Set up evaluation callback to monitor validation split and save only the best model
eval_callback = EvalCallback(
    val_env,
    best_model_save_path="./logs/best_model",
    log_path="./logs/results",
    eval_freq=5000,
    deterministic=True,
    render=False
)

print("Starting PPO Agent Policy Training (100,000 steps with validation callback)... (Logs will display below)")
model.learn(total_timesteps=100000, callback=eval_callback)

# Save or copy the best model to nifty_ppo_agent.zip
best_model_path = "./logs/best_model/best_model.zip"
if os.path.exists(best_model_path):
    shutil.copy(best_model_path, "nifty_ppo_agent.zip")
    print("Best validation model successfully copied to: nifty_ppo_agent.zip")
else:
    model.save("nifty_ppo_agent")
    print("Final model saved directly to: nifty_ppo_agent.zip")

### Step 6: Out-of-Sample Performance Evaluation
We evaluate the policy on the remaining 15% unseen chronological test data using the best validation checkpoint.

In [ ]:
import os
from stable_baselines3 import PPO

# Load the best saved model for final out-of-sample evaluation
if os.path.exists("nifty_ppo_agent.zip"):
    model = PPO.load("nifty_ppo_agent", env=test_env)
    print("Loaded the best PPO model for final testing!")

obs, _ = test_env.reset()
done = False

trades = []
wins, losses = 0, 0

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = test_env.step(action)
    done = terminated or truncated
    
    if 'trade' in info:
        net_trade_pnl = info['trade']['pnl'] * QTY
        trades.append(net_trade_pnl)
        if net_trade_pnl > 0:
            wins += 1
        else:
            losses += 1
trades = np.array(trades)
total_trades = len(trades)

print("=" * 60)
print("      RL PPO AGENT OUT-OF-SAMPLE BACKTEST RESULTS (BEST MODEL)")
print("=" * 60)
print(f"  Total Trades Taken : {total_trades}")
if total_trades > 0:
    win_rate = (wins / total_trades) * 100
    net_profit = trades.sum()
    print(f"  Win Rate           : {win_rate:.2f}%")
    print(f"  Net PnL (Rs.)      : Rs. {net_profit:,.2f}")
else:
    print("  Agent took 0 trades.")
print("=" * 60)

# Auto download model file back to local machine
from google.colab import files
try:
    files.download("nifty_ppo_agent.zip")
    print("Model nifty_ppo_agent.zip downloaded successfully.")
except Exception as e:
    print(f"Could not auto-download model (manually download nifty_ppo_agent.zip from files tab if needed): {e}")